In [ ]:


# Install correct versions (IMPORTANT)
!pip install -q langchain langchain-openai langchain-community faiss-cpu tiktoken requests==2.32.4

# Imports (ALL FIXED)
import os
from getpass import getpass

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain.tools import tool
from langchain.agents import initialize_agent

# API Key
os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")

# LLM
llm = ChatOpenAI(model="gpt-4o-mini")

print("\nLLM Ready\n")

# 1. BASIC LLM

print("Basic LLM:")
print(llm.invoke("Explain AI simply").content)


# 2. PROMPT + CHAIN (NEW STYLE)

print("\nPrompt + Chain:")

prompt = PromptTemplate.from_template("Explain {topic} in simple words")

chain = prompt | llm | StrOutputParser()

print(chain.invoke({"topic": "Machine Learning"}))


# 3. MEMORY (MANUAL SIMPLE VERSION)
print("\nMemory Example:")

chat_history = []

def chat(input_text):
    chat_history.append("User: " + input_text)
    response = llm.invoke("\n".join(chat_history)).content
    chat_history.append("AI: " + response)
    return response

print(chat("Hi"))
print(chat("What is AI?"))


# 4. AGENT + TOOL

print("\nAgent Example:")

@tool
def calculator(query: str) -> str:
    return str(eval(query))

agent = initialize_agent(
    tools=[calculator],
    llm=llm,
    agent="zero-shot-react-description",
    verbose=True
)

print(agent.run("What is 10 + 20 * 2?"))

# 5. VECTOR STORE

print("\nVector Store:")

texts = [
    "AI is future",
    "Machine Learning is part of AI",
    "Deep Learning uses neural networks"
]

embeddings = OpenAIEmbeddings()

db = FAISS.from_texts(texts, embeddings)

results = db.similarity_search("What is AI?")

for r in results:
    print(r.page_content)


# DONE

print("\nAll components working successfully")